In [2]:
import duckdb
con = duckdb.connect(r"C:\Users\lopec\OneDrive\Documentos\TFG_Selmark\duckdb\selmark.duckdb", read_only=True)

print("=" * 60)
print("CALIDAD DE LOS NUEVOS CAMPOS (con casteo)")
print("=" * 60)
stats = con.execute("""
    SELECT
        COUNT(*) AS total_filas,
        SUM(CASE WHEN TRY_CAST(importe_total AS DOUBLE) > 0 THEN 1 ELSE 0 END) AS importe_pos,
        SUM(CASE WHEN TRY_CAST(importe_total AS DOUBLE) = 0 THEN 1 ELSE 0 END) AS importe_cero,
        SUM(CASE WHEN importe_total IS NULL OR importe_total = '' THEN 1 ELSE 0 END) AS importe_nulo,
        SUM(CASE WHEN TRY_CAST(importe_total AS DOUBLE) IS NULL AND importe_total IS NOT NULL AND importe_total != '' THEN 1 ELSE 0 END) AS no_castea,
        ROUND(AVG(TRY_CAST(importe_total AS DOUBLE)), 2) AS media,
        ROUND(MIN(TRY_CAST(importe_total AS DOUBLE)), 2) AS minimo,
        ROUND(MAX(TRY_CAST(importe_total AS DOUBLE)), 2) AS maximo
    FROM bronze.ventas_minoristas
""").fetchdf()
print(stats.to_string(index=False))

print("\nMuestra de 5 filas con importe > 0:")
muestra = con.execute("""
    SELECT importe_total, importe_total_con_descuento
    FROM bronze.ventas_minoristas
    WHERE TRY_CAST(importe_total AS DOUBLE) > 0
    LIMIT 5
""").fetchdf()
print(muestra.to_string(index=False))

print("\nValores raros (si hay alguno que no castea):")
raros = con.execute("""
    SELECT DISTINCT importe_total
    FROM bronze.ventas_minoristas
    WHERE TRY_CAST(importe_total AS DOUBLE) IS NULL
      AND importe_total IS NOT NULL 
      AND importe_total != ''
    LIMIT 10
""").fetchdf()
print(raros.to_string(index=False) if len(raros) > 0 else "(ninguno, todo castea limpio)")

con.close()

CALIDAD DE LOS NUEVOS CAMPOS (con casteo)
 total_filas  importe_pos  importe_cero  importe_nulo  no_castea  media  minimo  maximo
     2220352          0.0           0.0           0.0  2220352.0    NaN     NaN     NaN

Muestra de 5 filas con importe > 0:
Empty DataFrame
Columns: [importe_total, importe_total_con_descuento]
Index: []

Valores raros (si hay alguno que no castea):
importe_total
         null


In [3]:
import duckdb
con = duckdb.connect(r"C:\Users\lopec\OneDrive\Documentos\TFG_Selmark\duckdb\selmark.duckdb", read_only=True)

print("=" * 60)
print("VALORES DISTINTOS EN importe_total")
print("=" * 60)
distintos = con.execute("""
    SELECT 
        importe_total,
        COUNT(*) AS cuantas_filas
    FROM bronze.ventas_minoristas
    GROUP BY importe_total
    ORDER BY cuantas_filas DESC
    LIMIT 20
""").fetchdf()
print(distintos.to_string(index=False))

print("\n¿Cuántos valores distintos hay en total?")
n_distintos = con.execute("SELECT COUNT(DISTINCT importe_total) FROM bronze.ventas_minoristas").fetchone()[0]
print(f"   {n_distintos:,} valores distintos")

con.close()

VALORES DISTINTOS EN importe_total
importe_total  cuantas_filas
         null        2220352

¿Cuántos valores distintos hay en total?
   1 valores distintos
